# construction-diff Demo

**construction-diff** detects construction progress by comparing point-cloud scans taken at different times.

## How it works

1. **Registration** -- Two scans captured at different epochs are aligned into a common coordinate frame using FPFH feature-based global registration (RANSAC) followed by ICP refinement.
2. **Multi-scale registration** -- For real-world scans with noise and varying scanner positions, a coarse-to-fine strategy processes progressively smaller voxel sizes. The coarse stage captures large structural features (walls, slabs) while fine stages recover detail.
3. **Diff computation** -- After alignment, KD-tree nearest-neighbour queries classify every point as *new* (construction added), *removed* (demolition / occlusion change), or *unchanged*.

## Use cases

- Tracking construction progress on a building site between weekly LiDAR scans.
- Detecting unauthorized modifications in an existing structure.
- Comparing as-built point clouds against a BIM reference.

This notebook walks through the full pipeline on synthetic data.

## 1. Create synthetic construction scenes

We build two point clouds:

- **Before**: A simple room with four walls and a floor (~5000 points).
- **After**: The same room with an additional interior wall section (~5000 points).

In [ ]:
import numpy as np
import open3d as o3d

np.random.seed(42)


def sample_wall(origin, size, n_points, noise_std=0.005):
    """Sample points on a rectangular wall surface.

    Parameters
    ----------
    origin : (3,) start corner
    size : (3,) extent along x, y, z -- one axis should be ~0 (thin wall)
    n_points : number of points to sample
    """
    pts = np.random.rand(n_points, 3) * size + origin
    pts += np.random.randn(n_points, 3) * noise_std
    return pts


# Room dimensions: 6m x 4m, walls 3m high, floor at z=0
ROOM_X, ROOM_Y, WALL_H = 6.0, 4.0, 3.0
WALL_THICKNESS = 0.05
N_WALL = 600
N_FLOOR = 1400

# -- Shared room geometry (walls + floor) --------------------------------
def make_room_points():
    walls = []
    # Front wall (y=0)
    walls.append(sample_wall([0, 0, 0], [ROOM_X, WALL_THICKNESS, WALL_H], N_WALL))
    # Back wall (y=ROOM_Y)
    walls.append(sample_wall([0, ROOM_Y, 0], [ROOM_X, WALL_THICKNESS, WALL_H], N_WALL))
    # Left wall (x=0)
    walls.append(sample_wall([0, 0, 0], [WALL_THICKNESS, ROOM_Y, WALL_H], N_WALL))
    # Right wall (x=ROOM_X)
    walls.append(sample_wall([ROOM_X, 0, 0], [WALL_THICKNESS, ROOM_Y, WALL_H], N_WALL))
    # Floor
    walls.append(sample_wall([0, 0, 0], [ROOM_X, ROOM_Y, WALL_THICKNESS], N_FLOOR))
    return np.vstack(walls)


# "Before" scene: bare room
before_pts = make_room_points()

# "After" scene: room + new interior wall (x=3, spanning y=[1,3], full height)
after_room = make_room_points()
new_wall = sample_wall([3.0, 1.0, 0.0], [WALL_THICKNESS, 2.0, WALL_H], 600)
after_pts = np.vstack([after_room, new_wall])

# Convert to Open3D point clouds
pcd_before = o3d.geometry.PointCloud()
pcd_before.points = o3d.utility.Vector3dVector(before_pts)

pcd_after = o3d.geometry.PointCloud()
pcd_after.points = o3d.utility.Vector3dVector(after_pts)

print(f"Before: {len(pcd_before.points):,} points")
print(f"After:  {len(pcd_after.points):,} points")

## 2. Registration

We compare single-scale `register_scans` with multi-scale `register_scans_multiscale`.

Since our synthetic scenes share the same coordinate frame, a perfect registration should yield an identity-like transformation with high fitness.

In [ ]:
from construction_diff.registration import register_scans, register_scans_multiscale

# Single-scale registration
result_single = register_scans(pcd_before, pcd_after, voxel_size=0.1)
print("=== Single-scale registration ===")
print(f"  Fitness:    {result_single.fitness:.4f}")
print(f"  Inlier RMSE: {result_single.inlier_rmse:.6f}")
print()

# Multi-scale registration
result_multi = register_scans_multiscale(
    pcd_before, pcd_after, voxel_sizes=[0.5, 0.2, 0.1]
)
print("=== Multi-scale registration ===")
print(f"  Fitness:    {result_multi.fitness:.4f}")
print(f"  Inlier RMSE: {result_multi.inlier_rmse:.6f}")
print()

print("Multi-scale transformation matrix:")
print(np.array2string(result_multi.transformation, precision=4, suppress_small=True))

## 3. Compute diff

Using the multi-scale result, we classify points as new, removed, or unchanged.

In [ ]:
from construction_diff.diff import compute_diff

diff = compute_diff(
    source=pcd_before,
    target=pcd_after,
    transformation=result_multi.transformation,
    threshold=0.10,
)

print(f"New points (construction added):  {diff['n_new']:>5,}")
print(f"Removed points:                   {diff['n_removed']:>5,}")
print(f"Unchanged points:                 {diff['n_unchanged']:>5,}")
print(f"Total target points:              {len(pcd_after.points):>5,}")

## 4. Visualization

Top-down (XY) and side (XZ) views with colour coding:
- **Green**: new points (added construction)
- **Red**: removed points
- **Gray**: unchanged

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from construction_diff.visualization import build_diff_cloud

diff_cloud = build_diff_cloud(
    pcd_before, pcd_after, diff, result_multi.transformation
)

pts = np.asarray(diff_cloud.points)
colors = np.asarray(diff_cloud.colors)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top-down view (XY)
ax = axes[0]
ax.scatter(pts[:, 0], pts[:, 1], c=colors, s=0.3, marker=".")
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_title("Top-down view (XY)")
ax.set_aspect("equal")

# Side view (XZ)
ax = axes[1]
ax.scatter(pts[:, 0], pts[:, 2], c=colors, s=0.3, marker=".")
ax.set_xlabel("X (m)")
ax.set_ylabel("Z (m)")
ax.set_title("Side view (XZ)")
ax.set_aspect("equal")

# Legend
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=(0, 0.8, 0), label="New", markersize=8),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=(0.9, 0, 0), label="Removed", markersize=8),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=(0.6, 0.6, 0.6), label="Unchanged", markersize=8),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=3, fontsize=11)
fig.suptitle("Construction Diff -- Synthetic Room", fontsize=14)
fig.tight_layout(rect=[0, 0.05, 1, 0.96])
plt.show()

## Summary

This demo showed the full **construction-diff** pipeline:

1. **Synthetic data** -- Two point-cloud scenes ("before" and "after") representing a room where a new interior wall was added.
2. **Registration** -- Both single-scale and multi-scale FPFH + ICP registration aligned the scans. Multi-scale registration is recommended for real-world data as it is more robust to noise and scanner position changes.
3. **Diff** -- KD-tree based nearest-neighbour classification correctly identified the ~600 new wall points as construction additions.
4. **Visualization** -- Matplotlib colour-coded plots clearly highlight the new wall section in green.

For real projects, replace the synthetic data with `.ply` or `.las` scans loaded via `construction_diff.loader`.